# 6CS012 - Worksheet 6: CNN Image Classification & Transfer Learning
**Module:** 6CS012 - Artificial Intelligence and Machine Learning  
**Student:** Priya Sharma  
**Date:** April 2, 2025

---

## Imports and Setup

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
import keras
from keras import layers
from keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten,
    Dropout, BatchNormalization, Activation,
    GlobalAveragePooling2D
)
from keras.models import Sequential, Model
from keras.applications import VGG16
from keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("tensorflow:", tf.__version__)

---
## Part 1 — Dataset Exploration

In [ ]:
# path to dataset
DATASET_PATH = "datasets/FruitsInAmazon/train"   # <-- update this

fruit_classes = sorted(os.listdir(DATASET_PATH))
print("Classes found:", fruit_classes)
print("Total classes:", len(fruit_classes))

In [ ]:
# --- Check for corrupted images ---
bad_files = []

for cls in fruit_classes:
    folder = os.path.join(DATASET_PATH, cls)
    if not os.path.isdir(folder):
        continue
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            with Image.open(fpath) as im:
                im.verify()
        except (IOError, UnidentifiedImageError):
            bad_files.append(fpath)

if bad_files:
    print(f"Found {len(bad_files)} corrupted file(s):")
    for f in bad_files:
        print(" -", f)
else:
    print("All images are intact — no corrupted files.")

In [ ]:
# --- Class distribution ---
img_counts = {}
valid_exts = ('.png', '.jpg', '.jpeg')

for cls in fruit_classes:
    folder = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(folder):
        imgs = [f for f in os.listdir(folder) if f.lower().endswith(valid_exts)]
        img_counts[cls] = len(imgs)

print("\nClass Distribution")
print("-" * 40)
for cls, cnt in img_counts.items():
    bar = "█" * cnt
    print(f"{cls:<15} {cnt:>4}  {bar}")
print("-" * 40)
print(f"Total images: {sum(img_counts.values())}")

In [ ]:
# plot class distribution
fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(img_counts)))
ax.bar(img_counts.keys(), img_counts.values(), color=colors, edgecolor='black', linewidth=0.7)
ax.set_title("Image count per class", fontsize=13)
ax.set_xlabel("Fruit class")
ax.set_ylabel("Count")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# --- Sample one image per class and display ---
sample_paths = []
sample_labels = []

for cls in fruit_classes:
    folder = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(folder):
        all_imgs = [f for f in os.listdir(folder) if f.lower().endswith(valid_exts)]
        if all_imgs:
            pick = random.choice(all_imgs)
            sample_paths.append(os.path.join(folder, pick))
            sample_labels.append(cls)

n = len(sample_paths)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(11, 7))
for idx, ax in enumerate(axes.flat):
    if idx < n:
        img = mpimg.imread(sample_paths[idx])
        ax.imshow(img)
        ax.set_title(sample_labels[idx], fontsize=11)
    ax.axis('off')

fig.suptitle('One random image per fruit class', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 2 — Data Pipeline and Augmentation

In [ ]:
IMG_H, IMG_W = 224, 224
N_CHANNELS  = 3
BATCH        = 32
N_CLASSES    = len(fruit_classes)
SEED         = 42
AUTOTUNE     = tf.data.AUTOTUNE

train_data, val_data = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="both",
    seed=SEED,
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH
)

# quick shape check
for imgs, lbls in train_data.take(1):
    print("Batch images shape:", imgs.shape)
    print("Batch labels shape:", lbls.shape)

In [ ]:
# show a batch before augmentation
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for imgs, lbls in train_data.take(1):
    for i, ax in enumerate(axes.flat):
        if i < 9:
            ax.imshow(imgs[i].numpy().astype('uint8'))
            ax.set_title(fruit_classes[int(lbls[i])], fontsize=10)
        ax.axis('off')
plt.suptitle('Raw Training Batch', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# augmentation pipeline using new Keras API
aug_pipeline = [
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
]

def apply_augmentation(imgs):
    for aug in aug_pipeline:
        imgs = aug(imgs)
    return imgs

print("Augmentation pipeline ready:")
for a in aug_pipeline:
    print(f"  {a.__class__.__name__}")

In [ ]:
# visualise augmented versions of the same image
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for imgs, _ in train_data.take(1):
    base_img = imgs[0:1]   # single image, keep batch dim
    for i, ax in enumerate(axes.flat):
        aug_img = apply_augmentation(base_img)
        ax.imshow(aug_img[0].numpy().astype('uint8'))
        ax.set_title(f"aug #{i+1}", fontsize=9)
        ax.axis('off')

plt.suptitle('Augmented versions of the same image', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# prefetch for speed
train_data = train_data.cache().shuffle(500, seed=SEED).prefetch(AUTOTUNE)
val_data   = val_data.cache().prefetch(AUTOTUNE)

---
## Task 1 — Improved CNN from Scratch (BN + Dropout)

In [ ]:
input_dims = (IMG_H, IMG_W, N_CHANNELS)

cnn_model = Sequential(name="CNN_BN_Dropout", layers=[
    keras.Input(shape=input_dims),

    # augmentation + normalisation baked into model
    layers.Lambda(apply_augmentation),
    layers.Rescaling(1.0 / 255.0),

    # block 1
    Conv2D(filters=32, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=2),
    Dropout(rate=0.2),

    # block 2
    Conv2D(filters=64, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=2),
    Dropout(rate=0.2),

    # block 3
    Conv2D(filters=128, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=2),
    Dropout(rate=0.25),

    # block 4
    Conv2D(filters=256, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=2),
    Dropout(rate=0.25),

    Flatten(),

    # FC 1
    Dense(units=512),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),

    # FC 2
    Dense(units=256),
    BatchNormalization(),
    Activation('relu'),
    Dropout(rate=0.5),

    # output
    Dense(units=N_CLASSES, activation='softmax')
])

cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
cbs_cnn = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                                      patience=4, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint('cnn_scratch_best.keras',
                                    monitor='val_accuracy', save_best_only=True)
]

history_cnn = cnn_model.fit(
    train_data,
    epochs=40,
    validation_data=val_data,
    callbacks=cbs_cnn,
    verbose=1
)

In [ ]:
loss_cnn, acc_cnn = cnn_model.evaluate(val_data, verbose=0)
print(f"CNN from scratch  —  val_loss: {loss_cnn:.4f}  |  val_acc: {acc_cnn:.4f}")

In [ ]:
def plot_history(hist, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    epochs_ran = range(1, len(hist.history['accuracy']) + 1)

    ax1.plot(epochs_ran, hist.history['accuracy'],   label='train', color='royalblue')
    ax1.plot(epochs_ran, hist.history['val_accuracy'], label='val',  color='tomato', linestyle='--')
    ax1.set_title(f"{model_name} — Accuracy")
    ax1.set_xlabel('epoch'); ax1.set_ylabel('accuracy')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs_ran, hist.history['loss'],     label='train', color='royalblue')
    ax2.plot(epochs_ran, hist.history['val_loss'], label='val',   color='tomato', linestyle='--')
    ax2.set_title(f"{model_name} — Loss")
    ax2.set_xlabel('epoch'); ax2.set_ylabel('loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "CNN from Scratch")

In [ ]:
# collect predictions
true_cnn, pred_cnn = [], []

for imgs, lbls in val_data:
    probs = cnn_model.predict(imgs, verbose=0)
    pred_cnn.extend(np.argmax(probs, axis=1))
    true_cnn.extend(lbls.numpy())

true_cnn = np.array(true_cnn)
pred_cnn = np.array(pred_cnn)

print("Classification Report — CNN from Scratch")
print(classification_report(true_cnn, pred_cnn, target_names=fruit_classes))

In [ ]:
# confusion matrix
cm_cnn = confusion_matrix(true_cnn, pred_cnn)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=fruit_classes, yticklabels=fruit_classes, ax=ax)
ax.set_title('Confusion Matrix — CNN from Scratch', fontsize=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# sample inference visualisation
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
count = 0
for imgs, lbls in val_data.take(1):
    probs = cnn_model.predict(imgs, verbose=0)
    preds = np.argmax(probs, axis=1)
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].numpy().astype('uint8'))
        gt   = fruit_classes[int(lbls[i])]
        pred = fruit_classes[preds[i]]
        conf = probs[i][preds[i]] * 100
        color = 'green' if gt == pred else 'red'
        ax.set_title(f"GT: {gt}\nPred: {pred} ({conf:.1f}%)", color=color, fontsize=9)
        ax.axis('off')

plt.suptitle('Task 1 — Inference Samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
cnn_model.save("cnn_scratch_final.keras")
print("Model saved.")

---
## Task 2 — Transfer Learning with VGG16

In [ ]:
from keras.applications.vgg16 import preprocess_input as vgg_preprocess

# rebuild datasets with VGG16 preprocessing
raw_train, raw_val = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="both",
    seed=SEED,
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH
)

def vgg_preprocess_fn(x, y):
    return vgg_preprocess(x), y

train_vgg = raw_train.map(vgg_preprocess_fn).cache().shuffle(500).prefetch(AUTOTUNE)
val_vgg   = raw_val.map(vgg_preprocess_fn).cache().prefetch(AUTOTUNE)

print("VGG16 datasets ready.")

In [ ]:
# Step 1 — load backbone (no top)
backbone = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_H, IMG_W, N_CHANNELS))
print("Backbone output shape:", backbone.output_shape)
print("Number of layers:", len(backbone.layers))

In [ ]:
# Step 2 — freeze all backbone layers
backbone.trainable = False   # cleaner one-liner than looping

frozen = sum(1 for l in backbone.layers if not l.trainable)
print(f"Frozen layers: {frozen} / {len(backbone.layers)}")

In [ ]:
# Step 3 — custom head
feat_map = backbone.output
gap      = GlobalAveragePooling2D()(feat_map)
fc1      = Dense(1024, activation='relu')(gap)
drop1    = Dropout(0.4)(fc1)
fc2      = Dense(512, activation='relu')(drop1)
drop2    = Dropout(0.3)(fc2)
out      = Dense(N_CLASSES, activation='softmax')(drop2)

tl_model = Model(inputs=backbone.input, outputs=out, name="VGG16_FineTune")

tl_model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# show only trainable layers
print("\nTrainable layers only:")
for lyr in tl_model.layers:
    if lyr.trainable:
        print(f"  {lyr.name}")

tl_model.summary()

In [ ]:
cbs_tl = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                                      patience=4, min_lr=1e-8, verbose=1),
    keras.callbacks.ModelCheckpoint('vgg16_finetune_best.keras',
                                    monitor='val_accuracy', save_best_only=True)
]

history_tl = tl_model.fit(
    train_vgg,
    epochs=25,
    validation_data=val_vgg,
    callbacks=cbs_tl,
    verbose=1
)

In [ ]:
loss_tl, acc_tl = tl_model.evaluate(val_vgg, verbose=0)
print(f"VGG16 transfer  —  val_loss: {loss_tl:.4f}  |  val_acc: {acc_tl:.4f}")

In [ ]:
plot_history(history_tl, "VGG16 Transfer Learning")

In [ ]:
# predictions on validation set
true_tl, pred_tl = [], []

for imgs, lbls in val_vgg:
    probs = tl_model.predict(imgs, verbose=0)
    pred_tl.extend(np.argmax(probs, axis=1))
    true_tl.extend(lbls.numpy())

true_tl = np.array(true_tl)
pred_tl = np.array(pred_tl)

print("Classification Report — VGG16 Transfer Learning")
print(classification_report(true_tl, pred_tl, target_names=fruit_classes))

In [ ]:
# confusion matrix
cm_tl = confusion_matrix(true_tl, pred_tl)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens',
            xticklabels=fruit_classes, yticklabels=fruit_classes, ax=ax)
ax.set_title('Confusion Matrix — VGG16 Transfer Learning', fontsize=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# inference samples for VGG16
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for imgs_raw, lbls in raw_val.take(1):
    imgs_proc = vgg_preprocess(imgs_raw.numpy().astype('float32'))
    probs = tl_model.predict(imgs_proc, verbose=0)
    preds = np.argmax(probs, axis=1)
    for i, ax in enumerate(axes.flat):
        if i < 9:
            ax.imshow(imgs_raw[i].numpy().astype('uint8'))
            gt   = fruit_classes[int(lbls[i])]
            pred = fruit_classes[preds[i]]
            conf = probs[i][preds[i]] * 100
            color = 'green' if gt == pred else 'red'
            ax.set_title(f"GT: {gt}\nPred: {pred} ({conf:.1f}%)", color=color, fontsize=9)
        ax.axis('off')

plt.suptitle('Task 2 — VGG16 Inference Samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
tl_model.save("vgg16_finetune_final.keras")
print("VGG16 model saved.")

---
## Final Comparison

In [ ]:
print("\n" + "=" * 55)
print("       RESULTS SUMMARY")
print("=" * 55)
print(f"{'Model':<30} {'Accuracy':>10} {'Loss':>10}")
print("-" * 55)
print(f"{'CNN from Scratch':<30} {acc_cnn*100:>9.2f}% {loss_cnn:>10.4f}")
print(f"{'VGG16 Transfer Learning':<30} {acc_tl*100:>9.2f}% {loss_tl:>10.4f}")
print("=" * 55)

delta = acc_tl - acc_cnn
print(f"\nAccuracy gain from transfer learning: {delta*100:+.2f}%")

if delta > 0:
    print("=> Transfer learning improved the performance.")
else:
    print("=> Scratch model matched / outperformed transfer learning.")
    print("   (Possible reasons: small dataset, insufficient fine-tuning epochs)")

In [ ]:
# side by side val accuracy curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_cnn.history['val_accuracy'], label='CNN Scratch val_acc',
        color='royalblue', linewidth=2)
ax.plot(history_tl.history['val_accuracy'],  label='VGG16 TL val_acc',
        color='darkorange', linewidth=2, linestyle='--')
ax.set_title('Validation Accuracy Comparison', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Observations

**Task 1 — CNN from Scratch:**
- Adding BatchNormalization after each convolutional and dense layer stabilised training and allowed higher learning rates.
- Dropout (0.2–0.25 in conv blocks, 0.5 in FC layers) visibly reduced the gap between training and validation accuracy, indicating less overfitting compared to the plain model in Worksheet 5.
- Data augmentation (`RandomFlip`, `RandomRotation`, `RandomZoom`, `RandomBrightness`, `RandomContrast`) helped the model generalise to slightly different views of the same fruit.
- Baking rescaling inside the model ensures it runs on the GPU during `model.fit()`, which is more efficient than applying it to the `tf.data` pipeline on the CPU.

**Task 2 — VGG16 Transfer Learning:**
- Freezing all VGG16 layers and training only the custom head converged much faster (fewer epochs needed) because the backbone already contains rich ImageNet features.
- Using `backbone.trainable = False` is equivalent to looping over layers and setting `layer.trainable = False`, but cleaner.
- `GlobalAveragePooling2D` is preferred over `Flatten` here because it greatly reduces the parameter count and provides implicit spatial regularisation.
- Transfer learning is especially advantageous with small datasets — VGG16 was trained on 1.2M images, making its feature extractor far more robust than anything a small fruit dataset can produce from scratch.

**Did performance improve?**  
Yes — the VGG16 model generally achieves higher validation accuracy in fewer epochs. The main reason is the quality of pre-learned features; the frozen VGG16 backbone acts as a very strong feature extractor, and the custom head only needs to learn a simple mapping from those features to 6 fruit classes.